In [5]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
texts = [
    ('../Data/True - True.csv',1),
    ('../Data/Fake - Fake.csv',0)
]

dfs = []

for file, label in texts:
    df_temp = pd.read_csv(file)
    df_temp['label'] = label
    dfs.append(df_temp)

df = pd.concat(dfs, axis=0)

df = df.sample(frac=1).reset_index(drop=True)

df.head()

,title,text,subject,date,label
0,Turkey's Erdogan says U.S. consulate hiding su...,ANKARA (Reuters) - Turkey s President Tayyip E...,worldnews,"October 12, 2017",1
1,Iraqi forces reach road between Iraqi-Syrian t...,BEIRUT (Reuters) - Iraqi forces have reached t...,worldnews,"November 3, 2017",1
2,"WHOA! “Canada’s Donald Trump,” Billionaire And...","Kevin O Leary, a rich Canadian television pers...",politics,"Jan 18, 2017",0
3,DISTURBING VIDEO: White Journalism Professor O...,An Asian student reporter on assignment for ES...,left-news,"Nov 10, 2015",0
4,UNHINGED TEXAS WOMAN DRIVING Pickup Truck With...,Just another case of Love Trumps Hate LOL! Th...,politics,"Nov 17, 2017",0


In [20]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = text.split()
    text = [word for word in text if word not in stop_words]
    return " ".join(text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hend\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')

df['content'] = df['title'] + " " + df['text']
df = df[['content', 'label']]

df['content'] = df['content'].apply(clean_text)

X = df['content']
y = df['label']

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
VOCAB_SIZE = 6000

tokenizer = Tokenizer(num_words=VOCAB_SIZE)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [26]:
max_len = 300

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [34]:
model = Sequential()

model.add(Embedding(VOCAB_SIZE, 128, input_length=max_len))
model.add(LSTM(128))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [33]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 136s 148ms/step - accuracy: 0.9520 - loss: 0.1415 - val_accuracy: 0.9839 - val_loss: 0.0569
Epoch 2/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 121s 135ms/step - accuracy: 0.9716 - loss: 0.0861 - val_accuracy: 0.7739 - val_loss: 0.8720
Epoch 3/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 149s 166ms/step - accuracy: 0.9663 - loss: 0.0942 - val_accuracy: 0.9857 - val_loss: 0.0444
Epoch 4/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 160s 178ms/step - accuracy: 0.9952 - loss: 0.0155 - val_accuracy: 0.9901 - val_loss: 0.0289
Epoch 5/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 161s 179ms/step - accuracy: 0.9981 - loss: 0.0065 - val_accuracy: 0.9890 - val_loss: 0.0448


In [35]:
y_pred = model.predict(X_test_pad)
y_pred = (y_pred > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

281/281 ━━━━━━━━━━━━━━━━━━━━ 19s 65ms/step
Accuracy: 0.4804008908685969
              precision    recall  f1-score   support

           0       0.51      0.43      0.46      4718
           1       0.46      0.54      0.50      4262

    accuracy                           0.48      8980
   macro avg       0.48      0.48      0.48      8980
weighted avg       0.48      0.48      0.48      8980



In [36]:
def predict_news(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)

    pred = model.predict(pad)[0][0]

    if pred > 0.5:
        return "TRUE NEWS"
    else:
        return "FAKE NEWS"